In [25]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
from agents.actor_delta_hedge import DeltaHedgeActor
from envs import BlackScholesEnv
import torch as T

# personal files
from models import PolicyApprox, ValueApprox
from risk_measure import RiskMeasure
from envs import BlackScholesEnv
from agents.actor_critic_pg import ActorCriticPG

# misc
import os
import argparse
import yaml

In [27]:
def setup_experiment(hyperparameters_version, is_training=False, preload=False):
    import yaml
    import os
    import torch as T
    from datetime import datetime

    # Paths & Device
    RUNS_DIR = "runs"
    os.makedirs(RUNS_DIR, exist_ok=True)
    device = T.device("cuda" if T.cuda.is_available() else "cpu")

    root_path = os.path.join(os.path.dirname(os.getcwd()))

    # Load hyperparameters
    with open(os.path.join(root_path,"hyperparameters.yml"), "r") as file:
        all_hyperparameter_sets = yaml.safe_load(file)
        hyperparameters = all_hyperparameter_sets[hyperparameters_version]

    envParams = hyperparameters["envParams"]
    algoParams = hyperparameters["algoParams"]
    riskParams = hyperparameters["riskParams"]
    runParams = hyperparameters["runParams"]
    repo_name = hyperparameters_version

    # Log message
    log_message = (
        f"*** Name of the repository:  {repo_name} ***\n"
        f"*** Environment parameters:  {envParams} ***\n"
        f"*** Algorithm parameters:  {algoParams} ***\n"
        f"*** Risk measures parameters:  {riskParams} ***\n"
        f"*** Run parameters:  {runParams} ***\n"
        f"*** Risk measures:  ['mean'] ***\n"
        f"*** alpha_cvar:  {[0.2]}"
    )
    print(log_message)

    # Assert Feller condition
    assert (
        2 * envParams["kappa"] * envParams["theta"] > envParams["eta"] ** 2
    ), "Feller condition is not satisfied."

    # Create repository
    repo = os.path.join(RUNS_DIR, repo_name)
    os.makedirs(repo, exist_ok=True)

    # Save log file
    LOG_FILE = os.path.join(repo, f"{hyperparameters_version}.log")
    with open(LOG_FILE, "w") as file:
        file.write(log_message + "\n")

    return {
        "device": device,
        "envParams": envParams,
        "algoParams": algoParams,
        "riskParams": riskParams,
        "runParams": runParams,
        "repo": repo,
        "log_file": LOG_FILE,
        "hyperparameters_version": hyperparameters_version,
        "is_training": is_training,
        "preload": preload,
    }


In [28]:
config = setup_experiment(
    hyperparameters_version="d002.000.001",
    is_training=True,
    preload=False
)

# Now you can pass `config` to build the env, agent, etc.


*** Name of the repository:  d002.000.001 ***
*** Environment parameters:  {'S0': 30, 'B0': 0, 'K': 30, 'T': 0.08333333, 'mu': 0.1, 'sigma': 0.02, 'r': 0.01, 'epsilon': 0, 'alpha0': 0, 'max_alpha': 8, 'Ndt': 31, 'kappa': 9, 'theta': 0.1, 'eta': 0.1, 'v0': 0.02} ***
*** Algorithm parameters:  {'Ntrajectories': 50, 'Mtransitions': 50, 'Nepochs': 40, 'gamma': 1, 'seed': 42, 'Nepochs_V_init': 150, 'Nepochs_V': 300, 'lr_V': 0.0005, 'batch_V': 20, 'hidden_V': 16, 'layers_V': 4, 'Nepochs_pi': 20, 'lr_pi': 0.0005, 'batch_pi': 20, 'hidden_pi': 16, 'layers_pi': 3} ***
*** Risk measures parameters:  {'method': 'mean'} ***
*** Run parameters:  None ***
*** Risk measures:  ['mean'] ***
*** alpha_cvar:  [0.2]


In [29]:
env = BlackScholesEnv(config["envParams"])
DH_agent = DeltaHedgeActor(
    os.getcwd(),
    "DH_actor",
    env,
    config["hyperparameters_version"],
    config["log_file"],
)


In [36]:
test_trajectories = DH_agent.sim_trajectories(
    1,
    5
)

In [31]:
test_trajectories

({'S': tensor([[ 30.0000,  32.9099,  37.1774,  40.9043,  45.3702,  48.1507,  54.5253,
            56.6618,  71.2953,  82.4356,  82.6500,  84.7231, 100.4504, 110.9298,
           120.4621, 128.6799, 148.8247, 167.7684, 169.1037, 195.8253, 198.3604,
           232.1781, 266.5732, 276.7410, 312.3630, 336.4552, 465.7767, 543.4089,
           447.2428, 529.5837, 472.5458]]),
  'v': tensor([[0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200,
           0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200,
           0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200, 0.0200,
           0.0200, 0.0200, 0.0200, 0.0200]]),
  'alpha': tensor([[ 0.0000, -7.8933,  6.5956,  1.2088, -0.0869,  1.6155, -4.5609, -1.2748,
            6.4453,  7.1366, -3.6321, -6.7768, -5.5299, -7.7187, -4.3178,  0.1910,
            5.0470, -2.7587,  6.6230, -6.6081, -7.0204, -3.0074, -6.1194, -5.4202,
           -2.6145, -3.1305,  3.8184,  2.4612,  1.2847, -6.0751,  3.3